<a href="https://colab.research.google.com/github/aspheth/simfolder/blob/master/14_case_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1

In [ ]:
def count_success_and_failure(file_path: str) -> tuple:
    total_attempts = 0
    failures = 0

    # Открываем файл в режиме чтения с указанием кодировки utf-8
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            # Проверяем, является ли строка инициацией обновления подписки
            if "[demon] Обновляем подписку пользователю id:" in line:
                total_attempts += 1

            # Проверяем, является ли строка фиксацией ошибки списания
            elif "ERROR" in line and "ошибка при списании" in line:
                failures += 1

    # Вычисляем количество успешных продлений
    successes = total_attempts - failures

    # Возвращаем кортеж из двух значений: (успешные, неуспешные)
    return (successes, failures)

In [ ]:
#2

In [ ]:
def calculate_median(lst):
    sorted_lst = sorted(lst)
    n = len(sorted_lst)
    if n % 2 != 0:
        return sorted_lst[n // 2]
    else:
        return (sorted_lst[n // 2 - 1] + sorted_lst[n // 2]) / 2

def auto_renewal_sub(file_path: str, output_path: str = "auto_renewal_sub.txt"):
    daily_data = {}

    # 1. Сбор данных и поиск максимума для каждого дня
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            if "количество людей с автопродлением подписки:" in line:
                parts = line.split(" | ")
                if len(parts) < 5:
                    continue

                # Достаем дату (ГГГГ-ММ-ДД)
                date_str = parts[1].split()[0].strip()

                # Достаем число
                num = int(line.split("количество людей с автопродлением подписки:")[1].strip())

                # Сохраняем только максимум для этой даты
                if date_str not in daily_data:
                    daily_data[date_str] = num
                else:
                    daily_data[date_str] = max(daily_data[date_str], num)

    # 2. СТРОГАЯ СОРТИРОВКА ДАТ ХРОНОЛОГИЧЕСКИ
    # Строки формата 'YYYY-MM-DD' сортируются стандартным sorted() абсолютно корректно
    sorted_dates = sorted(daily_data.keys())

    # Выстраиваем цепочку значений строго по датам
    chronological_values = [daily_data[date] for date in sorted_dates]

    moving_averages = []
    moving_medians = []
    current_history = []

    # 3. Расчет сглаживания «нарастающим итогом»
    for val in chronological_values:
        current_history.append(val)

        # Скользящее среднее (округляем до 2 знаков)
        avg = round(sum(current_history) / len(current_history), 2)
        moving_averages.append(avg)

        # Скользящая медиана
        med = calculate_median(current_history)
        if med == int(med):
            med = int(med)
        moving_medians.append(med)

    # 4. Запись в файл
    with open(output_path, 'w', encoding='utf-8') as out_file:
        out_file.write(f"Среднее: {moving_averages}\n")
        out_file.write(f"Медиана: {moving_medians}\n")

auto_renewal_sub('auto_purchase.log', 'auto_renewal_sub.txt')

In [ ]:
#3

In [ ]:
from datetime import datetime

def sub_renewal_by_day(file_path: str, output_path: str = "weekdays.txt"):
    # Словарь для сопоставления номеров дней (0-6) с русскими названиями
    days_names = {
        0: "Понедельник", 1: "Вторник", 2: "Среда",
        3: "Четверг", 4: "Пятница", 5: "Суббота", 6: "Воскресенье"
    }

    # Инициализируем счетчики для каждого дня недели
    days_stats = {i: 0 for i in range(7)}

    # Буфер для отслеживания активных попыток: {user_id: дата_строкой}
    active_attempts = {}

    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            parts = line.split(" | ")
            if len(parts) < 5:
                continue

            log_level = parts[0].strip()
            # Выделяем только дату YYYY-MM-DD из временной метки лога
            date_str = parts[1].split()[0].strip()
            content = parts[4]

            # 1. Если зафиксирована попытка обновления
            if "Обновляем подписку пользователю id:" in content:
                user_id = content.split("id:")[1].split(",")[0].strip()
                active_attempts[user_id] = date_str

            # 2. Если зафиксирована ошибка списания — убираем пользователя из успешных
            elif log_level == "ERROR" and "ошибка при списании" in content:
                user_id = content.split("id:")[1].split("-")[0].strip()
                if user_id in active_attempts:
                    del active_attempts[user_id]

    # 3. Считаем дни недели для всех оставшихся (успешных) продлений
    for user_id, date_str in active_attempts.items():
        dt = datetime.strptime(date_str, "%Y-%m-%d")
        day_of_week = dt.weekday()  # Понедельник = 0, Воскресенье = 6
        days_stats[day_of_week] += 1

    # 4. Записываем результат в файл строго в заданном формате
    with open(output_path, 'w', encoding='utf-8') as out_file:
        out_file.write("Количество обновлений подписки по дням недели:\n")
        for i in range(7):
            out_file.write(f"{days_names[i]}: {days_stats[i]}\n")